# R2-03 · Estadística para modelar — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 5–6 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Describir una distribución real (centro, dispersión, cola).
2. Estimar con **intervalos de confianza**.
3. Contrastar grupos con una **prueba de hipótesis** (t-test).
4. Medir **correlación** e interpretarla con cuidado.

**Competencia de salida:** usar estadística inferencial para sustentar decisiones de modelado.

**Dato real:** monto, cantidad y tamaño de proveedor de compras públicas (ChileCompra).

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

### Datos cargados arriba. A explorarlos con rigor.

## 1. Describir antes de modelar

Antes de cualquier modelo hay que **conocer la distribución**: centro, dispersión y cola. El monto de compras públicas tiene cola larga (pocas compras enormes).

### ✍️ Ejercicio 1 — Calcula el resumen descriptivo del monto

In [ ]:
x = df["monto_total"]
desc = {
    "media":   float(x.mean()),
    "mediana": float(x.median()),
    "p90":     float(x.quantile(0.90)),
    "std":     float(x.std()),
}
print(desc)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert set(desc) == {"media", "mediana", "p90", "std"}
    assert desc["p90"] >= desc["mediana"]
    assert desc["std"] > 0
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'x.mean(), x.median(), x.quantile(0.90) y x.std().')
    print("   Detalle:", e)

In [ ]:
# (ilustración) Distribución del monto (escala log por la cola larga)
plt.figure(figsize=(6, 4))
plt.hist(df["monto_total"].clip(upper=df["monto_total"].quantile(0.99)), bins=40, color="#4240e5")
plt.xlabel("Monto total (CLP, recortado al p99)"); plt.ylabel("Frecuencia")
plt.title("Distribución del monto de compra"); plt.tight_layout(); plt.show()

## 2. Intervalo de confianza para la media

La media de la muestra es un **estimado**. El **intervalo de confianza (95%)** dice en qué rango está, con incertidumbre, la media real.

### ✍️ Ejercicio 2 — Calcula el IC 95% de la media del monto

In [ ]:
x = df["monto_total"]
media = x.mean()
ic_low, ic_high = stats.t.interval(0.95, len(x) - 1, loc=media, scale=stats.sem(x))
print("media:", round(media), "| IC95%: [", round(ic_low), ",", round(ic_high), "]")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert ic_low < media < ic_high
    assert ic_high - ic_low > 0
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.t.interval(0.95, len(x)-1, loc=media, scale=stats.sem(x)) devuelve (low, high).')
    print("   Detalle:", e)

## 3. Prueba de hipótesis: ¿difieren dos grupos?

¿El monto de las compras a proveedores **Grandes** difiere del de los **Micro** más allá del azar? Un **t-test** lo responde: si el *p-valor* < 0.05, la diferencia es significativa.

### ✍️ Ejercicio 3 — Compara montos: Grande vs Micro

In [ ]:
grande = df[df["tamano_proveedor"] == "Grande"]["monto_total"]
micro  = df[df["tamano_proveedor"] == "Micro"]["monto_total"]
t_stat, p_valor = stats.ttest_ind(grande, micro, equal_var=False)
print("t:", round(t_stat, 2), "| p-valor:", p_valor)
print("¿Diferencia significativa al 5%?", p_valor < 0.05)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert 0.0 <= p_valor <= 1.0
    assert np.isfinite(t_stat)
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'stats.ttest_ind(grande, micro, equal_var=False) (test de Welch).')
    print("   Detalle:", e)

## 4. Correlación: ¿se mueven juntas dos variables?

¿A mayor **cantidad** de artículos, mayor **monto**? La correlación de Pearson mide la relación lineal (−1 a 1) y su *p-valor* dice si es significativa.

### ✍️ Ejercicio 4 — Correlación entre cantidad y monto

In [ ]:
r, p = stats.pearsonr(df["cantidad"], df["monto_total"])
print("r de Pearson:", round(r, 3), "| p-valor:", p)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert -1.0 <= r <= 1.0
    assert 0.0 <= p <= 1.0
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "stats.pearsonr(df['cantidad'], df['monto_total']) devuelve (coef, p_valor).")
    print("   Detalle:", e)

In [ ]:
# (ilustración) Cantidad vs monto (muestra, por la masa de puntos)
m = df.sample(min(800, len(df)), random_state=42)
plt.figure(figsize=(6, 4))
plt.scatter(m["cantidad"], m["monto_total"], s=10, alpha=0.4, color="#ff6b35")
plt.xlabel("Cantidad"); plt.ylabel("Monto total"); plt.title("Cantidad vs monto")
plt.tight_layout(); plt.show()

## Cierre

- La **descriptiva** (media, mediana, p90, dispersión) revela la forma del dato antes de modelar.
- Un **intervalo de confianza** acompaña cada estimación con su incertidumbre.
- Una **prueba de hipótesis** distingue una diferencia real del azar (p-valor < 0.05).
- La **correlación** sugiere relaciones, pero *correlación no es causalidad*.

> *Estadística just-in-time: la usamos para tomar decisiones honestas, no como teoría aparte.*